# Semantic Kernel

在这个代码示例中，您将使用 [Semantic Kernel](https://aka.ms/ai-agents-beginners/semantic-kernel) AI 框架创建一个基本的智能体。

这个示例的目标是向您展示我们稍后在实现不同智能体设计模式时将使用的步骤。

## 导入所需的 Python 包

In [1]:
import json
import os

from typing import Annotated

from dotenv import load_dotenv

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

## 创建客户端

在这个示例中，我们将使用 [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) 来访问 LLM。

`ai_model_id` 被定义为 `gpt-4o-mini`。尝试将模型更改为 GitHub Models 市场上可用的其他模型，以查看不同的结果。

为了使用 GitHub Models 的 `base_url` 所使用的 `Azure Inference SDK`，我们将使用 Semantic Kernel 中的 `OpenAIChatCompletion` 连接器。还有其他 [可用的连接器](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion) 可以使用 Semantic Kernel 来连接其他模型提供商。

In [2]:
import random   

# 定义一个示例插件

class DestinationsPlugin:
    """一个随机度假目的地列表。"""

    def __init__(self):
        # 度假目的地列表
        self.destinations = [
            "巴塞罗那, 西班牙",
            "巴黎, 法国",
            "柏林, 德国",
            "东京, 日本",
            "悉尼, 澳大利亚",
            "纽约, 美国",
            "开罗, 埃及",
            "开普敦, 南非",
            "里约热内卢, 巴西",
            "巴厘岛, 印度尼西亚"
        ]
        # 跟踪上次的目的地以避免重复
        self.last_destination = None

    @kernel_function(description="提供一个随机的度假目的地。")
    def get_random_destination(self) -> Annotated[str, "返回一个随机的度假目的地。"]:
        # 获取可用的目的地（如果可能，排除上次的目的地）
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # 选择一个随机目的地
        destination = random.choice(available_destinations)

        # 更新上次的目的地
        self.last_destination = destination

        return destination

In [6]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    # base_url="https://models.inference.ai.azure.com/",
    base_url=os.environ.get("API_URL"),
)

# 创建一个 AI 服务，将被 `ChatCompletionAgent` 使用
chat_completion_service = OpenAIChatCompletion(
    # ai_model_id="gpt-4o-mini",
    ai_model_id=os.environ.get("MODEL_FREE_8B"),
    async_client=client,
)

## 创建智能体

下面我们创建一个名为 `TravelAgent` 的智能体。

在这个示例中，我们使用非常简单的指令。您可以更改这些指令，以查看智能体如何以不同的方式响应。

In [7]:
AGENT_INSTRUCTIONS = """你是一个有用的 AI 智能体，可以帮助客户规划度假。

重要：当用户指定目的地时，始终为该位置规划。仅当用户未指定偏好时才建议随机目的地。

当对话开始时，用以下消息自我介绍：
"你好！我是你的 TravelAgent 助手。我可以帮助你规划度假并为你推荐有趣的目的地。你可以问我以下事情：
1. 为特定地点规划一日游
2. 推荐随机度假目的地
3. 寻找具有特定特色的目的地（海滩、山脉、历史遗迹等）
4. 如果不喜欢你的第一个建议，规划替代旅行

你今天想让我帮你规划什么样的旅行？"

始终优先考虑用户偏好。如果他们提到特定目的地，如"巴厘岛"或"巴黎"，请专注于该位置的规划，而不是建议替代方案。
"""

agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions=AGENT_INSTRUCTIONS,
)

## 运行智能体

现在我们可以通过定义 `ChatHistory` 并向其添加 `system_message` 来运行智能体。我们将使用我们之前定义的 `AGENT_INSTRUCTIONS`。

定义这些后，我们创建一个 `user_inputs`，这将是用户发送给智能体的内容。在这种情况下，我们将此消息设置为 `Plan me a sunny vacation`。

随意更改此消息，以查看智能体如何以不同的方式响应。

In [9]:
user_inputs = [
    "为我规划一次一日游。",
    "我不喜欢那个目的地。为我规划另一次度假。",
]

async def main():
    thread: ChatHistoryAgentThread | None = None

    # 首先，让智能体自我介绍
    agent_name = None
    full_response: list[str] = []
    function_calls: list[str] = []

    # 用于重建流式函数调用的缓冲区
    current_function_name = None
    argument_buffer = ""

    # 调用智能体，用简单的问候语触发介绍
    async for response in agent.invoke_stream(
        messages="Hello",
        thread=thread,
    ):
        thread = response.thread
        agent_name = response.name
        content_items = list(response.items)

        for item in content_items:
            if isinstance(item, FunctionCallContent):
                if item.function_name:
                    current_function_name = item.function_name

                # 累积参数（以块形式流式传输）
                if isinstance(item.arguments, str):
                    argument_buffer += item.arguments
            elif isinstance(item, FunctionResultContent):
                # 在显示结果之前完成任何待处理的函数调用
                if current_function_name:
                    formatted_args = argument_buffer.strip()
                    try:
                        parsed_args = json.loads(formatted_args)
                        formatted_args = json.dumps(parsed_args)
                    except Exception:
                        pass  # 保留为原始字符串

                    function_calls.append(f"调用函数: {current_function_name}({formatted_args})")
                    current_function_name = None
                    argument_buffer = ""

                function_calls.append(f"\n函数结果:\n\n{item.result}")
            elif isinstance(item, StreamingTextContent) and item.text:
                full_response.append(item.text)

    # 显示智能体介绍
    html_output = ""
    if function_calls:
        html_output += (
            "<div style='margin-bottom:10px'>"
            "<details>"
            "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>函数调用（点击展开）</summary>"
            "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
            "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
            f"{chr(10).join(function_calls)}"
            "</div></details></div>"
        )

    html_output += (
        "<div style='margin-bottom:20px'>"
        f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
        f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
    )

    display(HTML(html_output))

    # 现在处理用户输入
    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>用户:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # 用于重建流式函数调用的缓冲区
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # 累积参数（以块形式流式传输）
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # 在显示结果之前完成任何待处理的函数调用
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # 保留为原始字符串

                        function_calls.append(f"调用函数: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\n函数结果:\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>函数调用（点击展开）</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()